# Тема 4. Від запису до тексту

Ви відкрили Jupyter Notebook: файл, у якому пояснення чергуються з короткими програмами мовою Python. Програми цього файлу проводять запис навчальної передачі через етапи перехоплення з теми 4: від огляду ефіру до змісту повідомлення.

**Як працювати.** Jupyter Notebook складається з комірок: одні комірки містять текст, як ця, а інші програму. Клацніть першу комірку з програмою (вона розташована одразу під цим текстом) і натисніть Shift+Enter: комірка виконається, а під нею з’явиться результат. Так само виконуйте решту комірок по черзі, згори донизу.

In [ ]:
# Запис навчальної передачі зі сторінки теми 4. Програма читає запис як ряд чисел.
import os
import urllib.request
import wave

import matplotlib.pyplot as plt
import numpy as np

NAME = "tema-4-peredacha.wav"
ADDRESS = "https://korotkiyeugene.github.io/rozumna-syla-rer-chapter/t4/" + NAME
if not os.path.exists(NAME):          # файла ще немає поруч: програма бере його зі сторінки теми
    urllib.request.urlretrieve(ADDRESS, NAME)

with wave.open(NAME) as f:
    RATE = f.getframerate()           # скільки чисел припадає на секунду запису
    signal = np.frombuffer(f.readframes(f.getnframes()), dtype="<i2") / 32768


def uk(number):
    """Число з десятковою комою і щонайбільше двома знаками після неї."""
    return f"{number:.2f}".rstrip("0").rstrip(".").replace(".", ",")


print("Чисел за секунду:", RATE)
print("Тривалість запису, с:", uk(len(signal) / RATE))

## Етап 1. Огляд ефіру

Запис складається з чисел. Кожне число, відлік, показує значення сигналу в одну мить. На першому етапі приймач оглядає ефір: тут ефіром є весь запис. Наступна програма ділить запис на шматки по 0,01 с, рахує потужність сигналу в кожному шматку й малює її.

In [ ]:
PIECE = 0.01                          # тривалість одного шматка, с
size = round(RATE * PIECE)            # скільки відліків в одному шматку
pieces = signal[: len(signal) // size * size].reshape(-1, size)
power = (pieces ** 2).mean(axis=1)    # потужність шматка: середнє значення квадрата сигналу

plt.figure(figsize=(9, 3))
plt.plot(np.arange(len(power)) * PIECE, power)
plt.xlabel("Час, с")
plt.ylabel("Потужність")
plt.title("Потужність сигналу в записі")
plt.show()

## Етап 2. Виявлення

На рисунку спершу видно лише шум. Потім потужність різко зростає, а наприкінці знову спадає: це передача. Наступна програма знаходить передачу: вимірює потужність шуму на початку запису й шукає шматки, значно потужніші за шум.

In [ ]:
noise = power[: round(0.5 / PIECE)].mean()      # на початку запису є лише шум
loud = np.flatnonzero(power > 4 * noise)        # шматки, значно потужніші за шум
START = loud[0] * PIECE               # початок передачі, с
END = (loud[-1] + 1) * PIECE          # кінець передачі, с

print("Передача починається на секунді", uk(START))
print("Передача триває, с:", uk(END - START))

Це перші метадані: час початку передачі та її тривалість. Ці метадані стають відомі ще до того, як прочитано хоч один знак.

## Етап 3. Демодуляція

У передачі чергуються два звуки. Нижчий, 1000 Гц, означає 0. Вищий, 2000 Гц, означає 1. Кожен біт звучить 0,06 с. Наступна програма показує сигнал довкола межі між двома сусідніми бітами.

In [ ]:
BIT = 0.06                            # скільки секунд звучить один біт
per_bit = round(RATE * BIT)           # скільки відліків в одному біті
first = round(START * RATE)           # номер відліку, з якого починається передача
edge = first + 2 * per_bit            # межа між бітом номер 2 і наступним
around = round(0.004 * RATE)          # скільки відліків показати з кожного боку від межі

plt.figure(figsize=(9, 3))
plt.plot(np.arange(-around, around) / RATE * 1000, signal[edge - around: edge + around])
plt.axvline(0, color="red", linestyle="--")           # межа між двома бітами
plt.axhline(0, color="gray", linewidth=0.8)           # нуль, через який проходить сигнал
plt.xlabel("Час від межі між бітами, мс")
plt.ylabel("Сигнал")
plt.title("Ліворуч від червоної лінії вищий звук, праворуч нижчий")
plt.show()

Лінія ламана, бо на одне коливання вищого звуку припадає лише 4 відліки. Вищий звук коливається частіше, тому частіше проходить через нуль: на рисунку нуль позначає сіра лінія. Звук 1000 Гц за 0,06 с проходить через нуль 120 разів, а звук 2000 Гц 240 разів. Наступна програма рахує ці переходи в кожному біті. Якщо переходів більше ніж 180, біт дорівнює 1, якщо менше, біт дорівнює 0.

In [ ]:
BORDER = 180                          # межа між кількістю переходів двох звуків
count = round((END - START) / BIT)    # скільки бітів у передачі
windows = signal[first: first + count * per_bit].reshape(count, per_bit)
# Скільки разів сигнал проходить через нуль у кожному біті: стільки разів він змінює знак.
crossings = (np.diff(np.sign(windows), axis=1) != 0).sum(axis=1)
bits = (crossings > BORDER).astype(np.uint8)

print("Бітів у передачі:", count)
print("Перші 16 бітів:", "".join(str(bit) for bit in bits[:16]))

Вісім бітів складають байт, а байт позначає один знак: літеру, цифру чи пробіл. Відповідність чисел і знаків задає кодова таблиця. У цій передачі це таблиця Windows-1251: мова Python називає її cp1251.

In [ ]:
# Кожні вісім бітів стають одним байтом.
data = bytes(np.packbits(bits[: count // 8 * 8]).tolist())


def as_text(some_bytes):
    """Байти як знаки кодової таблиці. Невидимий знак замінює крапка посеред рядка."""
    signs = some_bytes.decode("cp1251", errors="replace")
    return "".join(sign if sign.isprintable() and sign != "\ufffd" else "·" for sign in signs)


print("Байтів у передачі:", len(data))
print(as_text(data))

Початок передачі читається без ключа. Це службова частина, яка називає, хто кого викликає. Далі стоять дивні знаки, бо зміст зашифровано.

## Етап 4. Дешифрування

Ключ цієї передачі є числом від 0 до 255. Кожен байт змісту поєднали з ключем біт за бітом дією «виключне або»: у мові Python її позначає знак `^`. Та сама дія з тим самим ключем повертає байт назад. Ключа ми не знаємо, зате ключів лише 256, і програма пробує всі. Справжній ключ дає текст, у якому є лише українські літери, цифри, пробіли й розділові знаки.

In [ ]:
# Скільки знаків має службова частина. Порахуйте їх у тексті під попередньою програмою.
HEADER = 14
secret = data[HEADER:]                # зашифрований зміст
LETTERS = "АБВГҐДЕЄЖЗИІЇЙКЛМНОПРСТУФХЦЧШЩЬЮЯ"
ALLOWED = set(LETTERS + LETTERS.lower() + "0123456789" + " .,:!?-")

for key in range(256):
    tried = as_text(bytes(byte ^ key for byte in secret))
    if all(sign in ALLOWED for sign in tried):
        print("Ключ", key, "дає текст:", tried)

## Етап 5. Контент-аналіз

Тепер зміст можна прочитати. Запишіть відповіді в зошит.

1. Хто кого викликав і про що повідомив?
2. Знайдіть цю передачу в журналі радіомережі з вправи теми 4: за номерами радіостанцій і тривалістю.
3. Що дізналася б розвідка з цієї передачі без ключа?

## Чому ключ має бути довгим

Ключ цієї передачі має 8 біт, тому ключів лише 256. Ключ сучасної мобільної мережі має 128 біт. Остання програма рахує, скільки є таких ключів і скільки років тривав би їх перебір.

In [ ]:
KEY_BITS = 128                        # скільки біт має ключ
PER_SECOND = 10 ** 9                  # скільки ключів за секунду пробує комп’ютер: тут мільярд
UNIVERSE = 13.8e9                     # вік Всесвіту, років


def spaced(number):
    """Велике ціле число з проміжками між трійками цифр."""
    return f"{number:,}".replace(",", " ")


keys = 2 ** KEY_BITS
years = keys // (PER_SECOND * 365 * 24 * 3600)
print("Ключів:", spaced(keys))
print("Років на перебір усіх ключів:", spaced(years))
print("Це довше, ніж існує Всесвіт, у стільки разів:", spaced(round(years / UNIVERSE)))

## Завдання

1. Замініть 128 у рядку `KEY_BITS = 128` на 129 і виконайте програму ще раз. У скільки разів зріс час перебору?
2. Поверніть 128. Уявіть комп’ютер, швидший у мільйон разів: замініть `10 ** 9` на `10 ** 15`. Чи став перебір можливим?
3. Розвідка не може дібрати такий ключ. Чому розвідка однаково приймає зашифровані передачі? Згадайте, що таке метадані.